# 09. Actor-Critic (A2C)

このノートブックでは **Advantage Actor-Critic (A2C)** アルゴリズムを実装・実行し、REINFORCE との学習安定性を比較します。

## 理論：Actor-Critic の仕組み

### REINFORCE の問題点

REINFORCE はシンプルですが、モンテカルロ法でリターンを推定するため **分散が大きく** 学習が不安定になりがちです。

### Actor と Critic の役割

| コンポーネント | 出力 | 役割 |
|---|---|---|
| **Actor** (方策) | `pi(a|s)` | 行動を選ぶ |
| **Critic** (価値関数) | `V(s)` | 状態の良さを評価し Actor にフィードバックする |

### Advantage 関数

Critic の価値推定を **ベースライン** として用いることで、分散を低減します：

$$A(s_t, a_t) = Q(s_t, a_t) - V(s_t) \approx G_t - V(s_t)$$

- $G_t$ : 時刻 $t$ からの割引リターン（モンテカルロ推定）
- $V(s_t)$ : Critic によるベースライン

### 更新則

$$\nabla_{\theta} J(\theta) \approx \sum_t A_t \cdot \nabla_{\theta} \log \pi_{\theta}(a_t | s_t)$$

$$\text{Critic loss} = \sum_t (G_t - V_{\phi}(s_t))^2$$

Actor と Critic は独立したネットワーク・独立した optimizer で更新します。

## セットアップ

In [ ]:
import sys
import os

# rlpg パッケージへのパスを追加
sys.path.insert(0, os.path.abspath('../src'))

import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from environments import InvertedPendulumEnv
from policies import ActorCriticPolicy, ReinforcePolicy
from utils.training import train_actor_critic, train_reinforce

print('Import OK')

In [ ]:
# 環境を作成
env = InvertedPendulumEnv(max_steps=500)
print(f'環境: {env}')
print(f'初期状態: {env.reset()}')

## ActorCriticPolicy の構築

パラメータ:

| パラメータ | デフォルト値 | 説明 |
|---|---|---|
| `hidden_sizes` | `[64, 64]` | Actor/Critic 共通の隠れ層サイズ |
| `action_low/high` | `-10 / +10` | 行動の値域 |
| `action_std` | `0.5` | ガウス探索の標準偏差 |
| `gamma` | `0.99` | 割引率 |
| `actor_lr` | `3e-4` | Actor の学習率 |
| `critic_lr` | `1e-3` | Critic の学習率（やや大きめ：早く収束させる） |

In [ ]:
# Actor-Critic ポリシーを構築
ac_policy = ActorCriticPolicy(
    hidden_sizes=[64, 64],
    action_low=-10.0,
    action_high=10.0,
    action_std=0.5,
    gamma=0.99,
    actor_lr=3e-4,
    critic_lr=1e-3,
)

print(f'Actor-Critic ポリシー: {ac_policy}')
print(f'  Actor  パラメータ数: {sum(p.numel() for p in ac_policy.actor_net.parameters())}')
print(f'  Critic パラメータ数: {sum(p.numel() for p in ac_policy.critic_net.parameters())}')
print(f'  合計パラメータ数   : {ac_policy.get_num_params()}')

## Actor-Critic の学習

In [ ]:
N_EPISODES = 300

print(f'Actor-Critic を {N_EPISODES} エピソード学習します...')
ac_result = train_actor_critic(
    policy=ac_policy,
    env=env,
    n_episodes=N_EPISODES,
    gamma=0.99,
    eval_every=50,
    verbose=True,
)

print(f'\n最終50エピソードの平均報酬: {np.mean(ac_result["reward_history"][-50:]):.1f}')

## 学習曲線の可視化

In [ ]:
def smooth(values, window=20):
    """移動平均でスムージング"""
    if len(values) < window:
        return values
    return np.convolve(values, np.ones(window) / window, mode='valid')


fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 報酬履歴 ---
ax = axes[0]
rewards = ac_result['reward_history']
ax.plot(rewards, alpha=0.3, color='steelblue', label='per episode')
ax.plot(
    range(len(smooth(rewards)) - 1 + 1, len(smooth(rewards)) + len(rewards) - len(smooth(rewards)) + 1),
    smooth(rewards),
    color='steelblue', linewidth=2, label='smoothed (window=20)'
)
# Fix x offset for smoothed
ax.clear()
ax.plot(rewards, alpha=0.3, color='steelblue', label='per episode')
smoothed = smooth(rewards)
offset = len(rewards) - len(smoothed)
ax.plot(range(offset, offset + len(smoothed)), smoothed, color='steelblue', linewidth=2, label='smoothed')
ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward')
ax.set_title('Actor-Critic: 学習曲線')
ax.legend()
ax.grid(True, alpha=0.3)

# --- Actor/Critic Loss ---
ax2 = axes[1]
ax2.plot(ac_result['actor_loss_history'], alpha=0.5, color='tomato', label='Actor loss')
ax2.plot(ac_result['critic_loss_history'], alpha=0.5, color='goldenrod', label='Critic loss')
ax2.set_xlabel('Episode')
ax2.set_ylabel('Loss')
ax2.set_title('Actor-Critic: 損失曲線')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('学習曲線を表示しました')

## REINFORCE との比較

同じ条件で REINFORCE も学習し、学習安定性を boxplot で比較します。

In [ ]:
# REINFORCE ポリシーを同条件で構築
reinforce_policy = ReinforcePolicy(
    hidden_sizes=[64, 64],
    action_low=-10.0,
    action_high=10.0,
    action_std=0.5,
    gamma=0.99,
    lr=3e-4,
)

print(f'REINFORCE ポリシー: {reinforce_policy}')
print(f'  パラメータ数: {reinforce_policy.get_num_params()}')

print(f'\nREINFORCE を {N_EPISODES} エピソード学習します...')
rf_result = train_reinforce(
    policy=reinforce_policy,
    env=env,
    n_episodes=N_EPISODES,
    gamma=0.99,
    eval_every=50,
    verbose=True,
)

print(f'\n最終50エピソードの平均報酬: {np.mean(rf_result["reward_history"][-50:]):.1f}')

In [ ]:
# 学習曲線の比較
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- 学習曲線（スムージング）比較 ---
ax = axes[0]
for label, result, color in [
    ('Actor-Critic (A2C)', ac_result, 'steelblue'),
    ('REINFORCE', rf_result, 'tomato'),
]:
    r = result['reward_history']
    s = smooth(r)
    off = len(r) - len(s)
    ax.plot(r, alpha=0.2, color=color)
    ax.plot(range(off, off + len(s)), s, color=color, linewidth=2, label=label)

ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward')
ax.set_title('学習曲線比較 (smoothed)')
ax.legend()
ax.grid(True, alpha=0.3)

# --- Boxplot（後半 150 エピソード）比較 ---
ax2 = axes[1]
last_n = 150
data_ac = ac_result['reward_history'][-last_n:]
data_rf = rf_result['reward_history'][-last_n:]

bp = ax2.boxplot(
    [data_ac, data_rf],
    labels=['Actor-Critic\n(A2C)', 'REINFORCE'],
    patch_artist=True,
    medianprops=dict(color='white', linewidth=2),
)

colors = ['steelblue', 'tomato']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax2.set_ylabel('Total Reward')
ax2.set_title(f'安定性比較 (最後の {last_n} エピソード)')
ax2.grid(True, alpha=0.3, axis='y')

# 統計情報を表示
for i, (label, data, color) in enumerate([
    ('A2C', data_ac, 'steelblue'),
    ('REINFORCE', data_rf, 'tomato')
]):
    print(f'{label:12s}: mean={np.mean(data):6.1f}  std={np.std(data):5.1f}  '
          f'median={np.median(data):6.1f}  max={np.max(data):6.1f}')

plt.tight_layout()
plt.show()

## まとめ

| 項目 | REINFORCE | Actor-Critic (A2C) |
|---|---|---|
| ベースライン | EMA（指数移動平均） | Critic ネットワーク V(s) |
| 分散 | 高い | 低い（Critic のおかげ） |
| バイアス | 低い（MC推定） | わずかにあり（Critic の近似誤差） |
| 学習安定性 | 低め | 高め |
| パラメータ数 | Actor のみ | Actor + Critic |

A2C では Critic が「どの状態が良いか」を学ぶことで、Actor の更新に使う Advantage の推定精度が上がり、学習が安定します。

次のステップとして、以下を試してみましょう：
- Critic の出力を Actor のネットワークと共有（パラメータ共有型 AC）
- 複数の環境を並列実行（Asynchronous AC / A3C）
- PPO（Proximal Policy Optimization）への発展